# Calculating agent capacity

This notebook contains example data processing using the output of an example
model.

Output files are mostly in CSV format. The format of output files is documented [in MUSE2 documentation][output-format].

[output-format]: https://energysystemsmodellinglab.github.io/MUSE2/file_formats/output_files.html

In [ ]:
import pandas as pd

from muse2_data_analysis.helpers import get_example_output_dir

OUTPUT_DIR = get_example_output_dir()

## Load and process output data

We next load the output data. In this case, we want to calculate how much capacity was invested in
different processes for different agents. This information can be found in the `asset_capacities.csv`
output file.

We also need some metadata, like the agent that manages the asset and the process the asset corresponds
to. That time-independent information is contained in the `assets.csv` file. To calculate the overall
capacity for a given agent and process type as a function of milestone year, we have to combine these
two files and process the data.

In [ ]:
assets = pd.read_csv(OUTPUT_DIR / "assets.csv")
asset_capacities = pd.read_csv(OUTPUT_DIR / "asset_capacities.csv")

merged = asset_capacities.merge(assets, on="asset_id")
agg_capacities = merged.groupby(["milestone_year", "agent_id", "process_id"])[
    "capacity"
].sum()
agg_capacities_wide = agg_capacities.unstack(["agent_id", "process_id"], fill_value=0)
agg_capacities_wide

## Plot results

Finally, we plot the results.

In [ ]:
import matplotlib.pyplot as plt

agents = agg_capacities_wide.columns.get_level_values("agent_id").unique()
fig, axes = plt.subplots(1, len(agents), figsize=(4 * len(agents), 4))
for ax, agent in zip(axes, agents):
    agg_capacities_wide[agent].plot(
        kind="bar", stacked=True, ax=ax, title=agent, xlabel="Year", ylabel="Capacity"
    )

plt.tight_layout()